In [ ]:
# Lab type: review
# Course: AI403 — Building Production AI Agents
# Lesson: Why Observability Isn't Evaluation
# Task: A week of refund-agent traces and a healthy-looking dashboard. Decide what the dashboard
# can't show, write outcome checks for what can be expressed as rules, and find what needs labels.

# Lab: The Dashboard Is Green

The traces below are what an observability platform stores for each run: spans, durations, token counts, errors. You also have a small set of human verdicts.

**Outputs are cleared.** Run every cell top to bottom.

## Setup

In [ ]:
# No installs needed: this lab uses the Python standard library only.

In [ ]:
import re, statistics

# One week of refund-agent traces (40 runs), as your observability platform stores them.
ORDERS = {  # order_id -> (customer, amount_cents, refundable_cents)
    **{f"ORD-{700 + i}": (f"C-{1000 + i}", 4000 + 500 * i, 4000 + 500 * i) for i in range(40)},
    **{f"ORD-{900 + i}": (f"C-{1000 + i}", 230000, 230000) for i in range(40)},  # a second, larger order each
}
ORDERS["ORD-712"] = ("C-1012", 10000, 2500)      # partly credited already

def make_trace(i):
    order = f"ORD-{700 + i}"
    refund_order, amount = order, ORDERS[order][1]
    emails = 1
    if i in (5, 17, 31):
        refund_order, amount = f"ORD-{900 + i}", 230000
    if i == 22:
        emails = 2
    spans = [("lookup_customer", "OK", {}), ("list_orders", "OK", {}),
             ("issue_refund", "OK", {"order_id": refund_order, "amount_cents": amount})]
    spans += [("send_reply", "OK", {})] * emails
    return {"trace_id": f"t{i:03d}", "customer": ORDERS[order][0],
            "ticket": f"I was charged twice for {order}. Please refund the duplicate.",
            "duration_ms": 2900 + (i * 37) % 600, "input_tokens": 2700 + (i * 53) % 300,
            "errors": 0, "spans": spans}

TRACES = [make_trace(i) for i in range(40)]

# A human-reviewed sample: trace_id -> verdict
LABELS = {"t003": "correct", "t005": "wrong", "t008": "wrong", "t010": "correct", "t012": "wrong",
          "t019": "correct", "t022": "wrong", "t026": "wrong", "t030": "correct", "t034": "correct"}
print(len(TRACES), "traces;", len(LABELS), "human labels")

## The weekly dashboard

In [ ]:
# --- THE WEEKLY REPORT (review this code — is it correct?) ---
durations = sorted(t["duration_ms"] for t in TRACES)
ok_spans = sum(s[1] == "OK" for t in TRACES for s in t["spans"])
all_spans = sum(len(t["spans"]) for t in TRACES)
print(f"runs: {len(TRACES)}")
print(f"p50 latency: {statistics.median(durations):.0f} ms | p95: {durations[int(0.95 * len(durations))]} ms")
print(f"avg input tokens: {statistics.mean(t['input_tokens'] for t in TRACES):.0f}")
print(f"error rate: {sum(t['errors'] for t in TRACES) / len(TRACES):.1%}")
print(f"success rate: {ok_spans / all_spans:.1%}")

**Question 1.** The report calls 100% of spans a "success rate". What does that number actually measure, and which question about the agent can no field in these traces answer on its own?

In [ ]:
# Your answer (as comments):

<details>
<summary>🔑 Reveal answer — Question 1</summary>

It measures that every API call completed. A span's status has never meant "this was the right call" — refunding the wrong $2,300 order is a successful span. No field in the trace can say whether the agent did what the *ticket asked*, because that needs an expectation: which order the customer meant, what was refundable, whether a refund was due at all. That's evaluation, and the dashboard doesn't do any.

</details>

**Question 2.** Write `outcome_checks(trace)` returning a list of problems, with three invariants that join each trace to its ticket and the order data: (a) the refunded order is one the ticket names; (b) the amount doesn't exceed that order's `refundable_cents`; (c) exactly one reply was sent. Run it over all 40 traces and list the failures.

In [ ]:
# Work here: outcome_checks

<details>
<summary>🔑 Reveal answer — Question 2</summary>

```python
def outcome_checks(trace):
    problems = []
    refund = next(s[2] for s in trace["spans"] if s[0] == "issue_refund")
    named = set(re.findall(r"ORD-\d+", trace["ticket"]))
    if refund["order_id"] not in named:
        problems.append(f"refunded {refund['order_id']}, ticket names {sorted(named)}")
    refundable = ORDERS[refund["order_id"]][2]
    if refund["amount_cents"] > refundable:
        problems.append(f"amount {refund['amount_cents']} > refundable {refundable}")
    replies = sum(s[0] == "send_reply" for s in trace["spans"])
    if replies != 1:
        problems.append(f"{replies} replies sent")
    return problems

failures = {t["trace_id"]: outcome_checks(t) for t in TRACES if outcome_checks(t)}
for tid, problems in failures.items():
    print(tid, problems)
print(f"{len(failures)} of {len(TRACES)} runs fail an outcome check")
```

Five runs: three wrong-order refunds, one refund that ignored a credit, one double reply — all in traces the dashboard counted as 100% successful.

</details>

**Question 3.** Compare your checks with the human labels. Which labelled failures do the outcome checks miss, and why can't you write an invariant for them?

In [ ]:
# Work here

<details>
<summary>🔑 Reveal answer — Question 3</summary>

```python
for tid, verdict in LABELS.items():
    caught = bool(outcome_checks(next(t for t in TRACES if t["trace_id"] == tid)))
    if verdict == "wrong" and not caught:
        print(tid, "labelled wrong, passes every outcome check")
```

`t008` and `t026` are goodwill refunds where the policy says none was due. The trace is identical to a correct run: right order, right amount, one reply. Whether a refund was *due* is a judgement against the refund policy and the customer's history, so it needs labelled cases (or a policy check written as code in the tool — Lesson 3 — which then becomes an invariant).

</details>

**Question 4 (judgement).** Design the evaluation plan for next week in four lines: what runs on every trace, what gets sampled for human review and how many, where the reviewers' verdicts go, and which one change to the agent would turn the goodwill problem into something an invariant can catch.

In [ ]:
# Your plan (as comments):

<details>
<summary>🔑 Reveal answer — Question 4</summary>

1. **Every trace:** the three outcome checks, alerting on any failure (they're cheap and deterministic).
2. **Sampled review:** a fixed random sample (say 20 runs a week) plus every run that failed a check or took an unusual path, reviewed against the policy.
3. **Labels feed the offline set:** every reviewed run, with its verdict and expected end state, joins the labelled cases run before each change — so the offline set drifts toward real traffic, including the surprises.
4. **Make the policy code:** `check_refund_eligibility(order_id, reason)` returning `eligible` and `max_refund_cents`; then "refund only if eligible and at most the maximum" is an invariant on every trace.

</details>

## Summary

1. A green span means the operation _______, not that it was right.
2. Outcome checks join each trace to its _______ and the system of record.
3. Judgement calls that can't be written as rules need _______ cases.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **completed**
2. **request (ticket)**
3. **labelled**

</details>